# 09 — Commercial Density

Computes shop-specific density and diversity metrics per census tract from OSM.

**Data source:** Overpass API (OSM `shop=*`) — fully portable.

**Distinct from notebook 02:** This focuses exclusively on shops (retail/commercial), computing shop-specific metrics like type entropy and brand ratio. Notebook 02 covers all amenity categories broadly.

**Output columns:** `tract_id`, `shop_count`, `shop_density_km2`, `shop_type_entropy`, `brand_ratio`

**Output file:** `csv/09_commercial_density.csv`

In [ ]:
ZONES_CONFIG = "zones.json"
QUERY_RADIUS = 500

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
import math
import hashlib
from collections import Counter

os.makedirs("csv", exist_ok=True)
os.makedirs("cache", exist_ok=True)

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")

In [ ]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "zone-finding/1.0 (research project)"}


def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            time.sleep(3 + attempt * 2)
    raise RuntimeError(f"Overpass failed: {last_error}")


def shannon_entropy(counts):
    """Shannon entropy from a Counter of type → count."""
    total = sum(counts.values())
    if total == 0:
        return 0.0
    proportions = np.array([c / total for c in counts.values()])
    proportions = proportions[proportions > 0]
    return -np.sum(proportions * np.log2(proportions))


print("Helpers ready.")

In [ ]:
# ── Query shops per tract ─────────────────────────────

AREA_KM2 = math.pi * (QUERY_RADIUS / 1000) ** 2
records = []
n_tracts = len(df_tracts)

for i, row in df_tracts.iterrows():
    tract_id = row["tract_id"]
    lat, lon = row["tract_lat"], row["tract_lon"]
    
    if (i + 1) % 25 == 0 or i == 0:
        print(f"  [{i+1}/{n_tracts}] Tract {tract_id}")
    
    query = (f'[out:json][timeout:30];\n'
             f'(node["shop"](around:{QUERY_RADIUS},{lat},{lon});\n'
             f' way["shop"](around:{QUERY_RADIUS},{lat},{lon}););\n'
             f'out center tags;')
    
    try:
        data = query_overpass_cached(query)
    except Exception as e:
        print(f"  ERROR tract {tract_id}: {e}")
        records.append({"tract_id": tract_id, "shop_count": 0,
                        "shop_density_km2": 0.0, "shop_type_entropy": 0.0, "brand_ratio": 0.0})
        continue
    
    shop_types = Counter()
    brand_count = 0
    total_shops = 0
    
    for el in data.get("elements", []):
        tags = el.get("tags", {})
        shop_val = tags.get("shop", "")
        if shop_val and shop_val not in {"vacant", "yes"}:
            shop_types[shop_val] += 1
            total_shops += 1
            if tags.get("brand"):
                brand_count += 1
    
    records.append({
        "tract_id": tract_id,
        "shop_count": total_shops,
        "shop_density_km2": round(total_shops / AREA_KM2, 2),
        "shop_type_entropy": round(shannon_entropy(shop_types), 4),
        "brand_ratio": round(brand_count / total_shops, 4) if total_shops > 0 else 0.0,
    })
    
    time.sleep(0.5)

df_shops = pd.DataFrame(records)
print(f"\nCompleted: {len(df_shops)} tracts")
print(f"Mean shops per tract: {df_shops['shop_count'].mean():.1f}")
print(f"Tracts with zero shops: {(df_shops['shop_count'] == 0).sum()}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/09_commercial_density.csv"
df_shops.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_shops)} rows x {df_shops.shape[1]} cols)")
print(df_shops.describe().round(2).to_string())